# Phase 10 — Robust Envelope Training
**RealCentric Generator-Agnostic Deepfake Detection**  
SVKM AI/ML HPC Cluster

Augments the real-image training data with JPEG, Blur, and Resize transformations and rebuilds the `DualOneClassEnsemble` real envelope on the full 981-dim normalised features.
This fixes the t-calibration for the smooth arm and corrects scorer inversion natively.

**Output:** `/data/mpstme-naman/deepfake_detection/checkpoints/ensemble_robust.pkl` (copied to `ensemble.pkl` for production)

## Step 1 — Load Images & Augment

In [1]:
import sys, numpy as np, time, cv2
import matplotlib.pyplot as plt
sys.path.insert(0, '/data/mpstme-naman/deepfake_detection')
from pathlib import Path
from tqdm import tqdm

BASE     = Path('/data/mpstme-naman/deepfake_detection')
PROC     = BASE / 'data' / 'processed'
FEAT_DIR = BASE / 'data' / 'features'
CKPT_DIR = BASE / 'checkpoints'
RES_DIR  = BASE / 'results'

real_paths = sorted((PROC/'celebdf'/'real').glob('*.png'))[:10000]
print(f'Augmenting {len(real_paths):,} real images from CelebDF...')

def augment_image(img):
    out = []
    for q in [90, 70, 50, 30]:
        _, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, q])
        out.append(cv2.imdecode(enc, cv2.IMREAD_COLOR))
    for k in [3, 7]:
        out.append(cv2.GaussianBlur(img, (k, k), 0))
    h, w = img.shape[:2]
    for scale in [0.75, 0.50]:
        small = cv2.resize(img, (int(w*scale), int(h*scale)))
        out.append(cv2.resize(small, (w, h)))
    return out

all_aug_imgs = []
for p in tqdm(real_paths, desc="Augmenting"):
    img = cv2.imread(str(p))
    if img is not None:
        all_aug_imgs.extend(augment_image(img))
print(f'\n  Generated {len(all_aug_imgs):,} augmented images.')

Augmenting 10,000 real images from CelebDF...


Augmenting: 100%|██████████| 10000/10000 [00:38<00:00, 263.08it/s]


  Generated 80,000 augmented images.


## Step 2 — Extract Features via Unified Pipeline
Uses `normalise=True` to inherit the baseline pipeline z-scoring.

In [2]:
from config.config_loader import load_config
from src.features.extractor import FeatureFusionPipeline

cfg = load_config()
pipeline = FeatureFusionPipeline(cfg=cfg, backbone=cfg['features']['cnn']['backbones'][0])
pipeline.set_state(np.load(CKPT_DIR / 'pipeline_state.pkl', allow_pickle=True))

print('Extracting features for augmented images (normalise=True)...')
t0 = time.time()
Z_aug = pipeline.extract_batch(all_aug_imgs, normalise=True, cnn_batch_size=64)
print(f'  ✓ Done in {time.time()-t0:.1f}s')
print(f'  Z_aug shape: {Z_aug.shape}')
del all_aug_imgs

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  CNN Backbone: GPU — NVIDIA H100 PCIe MIG 3g.40gb  (42.4 GB)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 54485.70it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
vision_model.embeddings.position_ids                         | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEX

Extracting features for augmented images (normalise=True)...
  ✓ Done in 1450.0s
  Z_aug shape: (80000, 981)


## Step 3 — Compile Robust Real Feature Set

In [3]:
# Load pre-extracted training features
Z_train = np.load(FEAT_DIR/'Z_train.npy')
y_train = np.load(FEAT_DIR/'y_train.npy')

Z_real_train = Z_train[y_train == 0]
print(f'  Original real features: {Z_real_train.shape}')

# Combine original reals with augmented reals
Z_robust_real = np.vstack([Z_real_train, Z_aug])
print(f'  Combined Robust Real Ensemble basis: {Z_robust_real.shape}')

  Original real features: (20368, 981)
  Combined Robust Real Ensemble basis: (100368, 981)


## Step 4 — Train DualOneClassEnsemble

In [4]:
from src.models.one_class_ensemble import DualOneClassEnsemble

ensemble = DualOneClassEnsemble(cfg=cfg)

# Fit ensemble scorers on combined (original + augmented) real 981-dim vectors
print('\nFitting Robust Envelope...')
ensemble.fit_envelope(Z_robust_real)


Fitting Robust Envelope...
Fitting Shared Envelope Components...
  Fitting envelope on 100,368 real samples (981 dims)...
    PCA Reconstruction scorer... done
    Mahalanobis scorer...        done
    Isolation Forest...          done
  Envelope fitted.
  Raw score ranges p5→p95 (real train): pca=[91.126,279.123]  mah=[88.695,186.415]  iso=[0.419,0.462]


## Step 5 — Calibrate Thresholds (Inversion Checks)
Explicitly checks and flips inverted scorers natively.

In [5]:
Z_val = np.load(FEAT_DIR/'Z_val.npy')
y_val = np.load(FEAT_DIR/'y_val.npy')
Z_ff = np.load(FEAT_DIR/'Z_ff.npy')
y_ff = np.load(FEAT_DIR/'y_ff.npy')

Z_real_val      = Z_val[y_val == 0]
Z_fake_celebdf  = Z_val[y_val == 1]
Z_fake_ff       = Z_ff[y_ff == 1]

print('\nCalibrating Robust Ensemble with inversion detection...')
tau_s, tau_a = ensemble.calibrate_threshold(
    val_real_features=Z_real_val,
    val_fake_smooth=Z_fake_celebdf,
    val_fake_artifact=Z_fake_ff,
    fpr_target=0.025
)
print(f'\n  \u2713  Smooth \u03c4 = {tau_s:.4f}  |  Artifact \u03c4 = {tau_a:.4f}')


Calibrating Robust Ensemble with inversion detection...

=== Calibrating SMOOTH Fake Ensemble (e.g. CelebDF) ===
  Per-scorer inversion check:
    pca_recon          correct  ✓  real_mean=200.7987  fake_mean=227.2034
    mahalanobis        FLIPPED ↩  real_mean=125.3156  fake_mean=117.7276
    isoforest          FLIPPED ↩  real_mean=0.4370  fake_mean=0.4341

  Threshold τ = 0.6387  (FPR target=2%  percentile=98th)

=== Calibrating ARTIFACT Fake Ensemble (e.g. FF++) ===
  Per-scorer inversion check:
    pca_recon          correct  ✓  real_mean=200.7987  fake_mean=294.2329
    mahalanobis        FLIPPED ↩  real_mean=125.3156  fake_mean=120.3018
    isoforest          correct  ✓  real_mean=0.4370  fake_mean=0.4373

  Threshold τ = 0.5615  (FPR target=2%  percentile=98th)

  ✓  Smooth τ = 0.6387  |  Artifact τ = 0.5615


## Step 6 — Save Checkpoints

In [6]:
import shutil

path_robust = CKPT_DIR / 'ensemble_robust.pkl'
path_prod   = CKPT_DIR / 'ensemble.pkl'

ensemble.save(str(path_robust))
shutil.copy(str(path_robust), str(path_prod))

print(f'\n  \u2713  Saved robust snapshot: {path_robust}')
print(f'  \u2713  Promoted to production : {path_prod}')

  DualEnsemble saved → /data/mpstme-naman/deepfake_detection/checkpoints/ensemble_robust.pkl

  ✓  Saved robust snapshot: /data/mpstme-naman/deepfake_detection/checkpoints/ensemble_robust.pkl
  ✓  Promoted to production : /data/mpstme-naman/deepfake_detection/checkpoints/ensemble.pkl


## Step 7 — Evaluate Sub-Arms (Verification)

In [7]:
Z_test = np.load(FEAT_DIR/'Z_test.npy')
y_test = np.load(FEAT_DIR/'y_test.npy')

print('=' * 50)
print('  Robust DualOneClassEnsemble Evaluation')
print('=' * 50)

m_cd = ensemble.evaluate(Z_test, y_test)
print(f'Overall CelebDF AUC : {m_cd["auc"]:.4f}')

try:
    # Test Smooth Arm implicitly
    cd_smooth = ensemble.ensemble_smooth.evaluate(Z_test, y_test)
    print(f'Smooth Arm CelebDF  : {cd_smooth["auc"]:.4f}  (baseline: 0.4986)')
except:
    pass

try:
    # Test Artifact Arm implicitly
    ff_artifact = ensemble.ensemble_artifact.evaluate(Z_ff, y_ff)
    print(f'Artifact Arm FF++   : {ff_artifact["auc"]:.4f}')
except:
    pass


  Robust DualOneClassEnsemble Evaluation
Overall CelebDF AUC : 0.6618
Smooth Arm CelebDF  : 0.6573  (baseline: 0.4986)
Artifact Arm FF++   : 0.5272
